# 12 - Pipelines
## Practical scikit-learn workflows

This notebook covers `Pipeline`, `ColumnTransformer`, missing-value handling, categorical encoding, cross-validation, hyperparameter tuning, feature names, and saving a fitted pipeline.

**Requirements:** `numpy`, `pandas`, `scikit-learn`, `joblib`.

Install if needed: `python -m pip install numpy pandas scikit-learn joblib`


In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_iris, load_breast_cancer, make_regression
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold, cross_val_score,
    cross_validate, GridSearchCV
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, classification_report

print("scikit-learn version:", sklearn.__version__)


## 1. The simplest pipeline

A pipeline chains transformers and a final estimator. Fitting and predicting then use one consistent object.

In [ ]:
X_iris, y_iris = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.2, stratify=y_iris, random_state=42
)

iris_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
iris_pipe.fit(X_train, y_train)
pred = iris_pipe.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, pred))
print(iris_pipe)


## 2. Named steps and parameter access

Use `Pipeline` when explicit step names are useful, especially for hyperparameter tuning.

In [ ]:
named_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
X_bc, y_bc = load_breast_cancer(return_X_y=True)
named_pipe.fit(X_bc, y_bc)
print("Steps:", list(named_pipe.named_steps))
print("Classifier C:", named_pipe.get_params()["model__C"])


## 3. Build a mixed-type dataset

Real-world DataFrames often contain numeric and categorical fields. We will deliberately include missing values so the preprocessing pipeline must handle them.

In [ ]:
rng = np.random.default_rng(42)
n = 240
customers = pd.DataFrame({
    "age": rng.integers(18, 75, n).astype(float),
    "income": rng.normal(60000, 18000, n),
    "city": rng.choice(["Indore", "Pune", "Delhi"], n),
    "plan": rng.choice(["basic", "plus", "premium"], n),
})
customers.loc[rng.choice(n, 18, replace=False), "age"] = np.nan
customers.loc[rng.choice(n, 16, replace=False), "income"] = np.nan
customers.loc[rng.choice(n, 14, replace=False), "city"] = None

target = (
    (customers["income"].fillna(60000) > 62000)
    & (customers["plan"].isin(["plus", "premium"]))
).astype(int)

display(customers.head())
print("Missing values per column:\n", customers.isna().sum())
print("Target distribution:\n", target.value_counts(normalize=True).round(3))


## 4. ColumnTransformer + Pipeline

Numeric features use median imputation and scaling. Categorical features use most-frequent imputation and one-hot encoding.

In [ ]:
numeric_features = ["age", "income"]
categorical_features = ["city", "plan"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

customer_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

X_train, X_test, y_train, y_test = train_test_split(
    customers, target, test_size=0.2, stratify=target, random_state=42
)
customer_model.fit(X_train, y_train)
print("Test accuracy:", accuracy_score(y_test, customer_model.predict(X_test)))
print(classification_report(y_test, customer_model.predict(X_test), zero_division=0))


## 5. Cross-validation on the complete pipeline

The preprocessor is refitted separately inside every fold, helping prevent data leakage.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(customer_model, customers, target, cv=cv, scoring="f1")
print("Fold F1:", np.round(scores, 3))
print(f"Mean F1: {scores.mean():.3f} ± {scores.std():.3f}")


## 6. Tune preprocessing and model parameters together

Pipeline parameters are addressed with `step_name__parameter_name`.

In [ ]:
param_grid = {
    "preprocessor__numeric__scaler": [StandardScaler(), MinMaxScaler()],
    "preprocessor__numeric__imputer__strategy": ["mean", "median"],
    "model__C": [0.1, 1.0, 10.0],
}
search = GridSearchCV(
    customer_model, param_grid=param_grid, scoring="f1",
    cv=cv, n_jobs=-1, return_train_score=True
)
search.fit(customers, target)
print("Best parameters:", search.best_params_)
print("Best CV F1:", round(search.best_score_, 3))


## 7. Inspect transformed feature names

Feature names can help debug encoding and connect transformed columns to the source schema.

In [ ]:
best_preprocessor = search.best_estimator_.named_steps["preprocessor"]
feature_names = best_preprocessor.get_feature_names_out()
print("Number of output features:", len(feature_names))
print(feature_names)


## 8. Handle categories not seen during fitting

`handle_unknown="ignore"` prevents a transform-time error for unseen categories. It does not guarantee that the model has learned a meaningful effect for the new category.

In [ ]:
new_customers = pd.DataFrame({
    "age": [31.0, np.nan],
    "income": [72000.0, 45000.0],
    "city": ["Indore", "NewCity"],
    "plan": ["premium", "basic"],
})
new_predictions = search.best_estimator_.predict(new_customers)
print("Predictions:", new_predictions)


## 9. Regression pipeline

Pipelines work for regression too. Scikit-learn's negative MAE scorer is negated here to report positive loss.

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=300, n_features=8, noise=15, random_state=42
)
regression_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Ridge(alpha=1.0),
)
reg_cv = KFold(n_splits=5, shuffle=True, random_state=42)
neg_mae = cross_val_score(
    regression_pipe, X_reg, y_reg, cv=reg_cv,
    scoring="neg_mean_absolute_error"
)
mae = -neg_mae
print("MAE per fold:", np.round(mae, 2))
print(f"Mean MAE: {mae.mean():.2f} ± {mae.std():.2f}")


## 10. Save and reload a fitted pipeline

Only load joblib/pickle artifacts from trusted sources because deserialization can execute code.

In [ ]:
import joblib
import tempfile
from pathlib import Path

fitted = search.best_estimator_
before = fitted.predict(new_customers)
artifact_path = Path(tempfile.gettempdir()) / "customer_pipeline_demo.joblib"
joblib.dump(fitted, artifact_path)
loaded = joblib.load(artifact_path)
after = loaded.predict(new_customers)
print("Predictions match after reload:", np.array_equal(before, after))
print("Saved demo artifact to:", artifact_path)


## Key takeaways

- Keep learned preprocessing inside the pipeline.
- Use `ColumnTransformer` for different feature types.
- Tune preprocessing and model parameters together with `GridSearchCV`.
- Use the same fitted pipeline at inference time.
- Keep final test data untouched during model selection.
- Persist model artifacts carefully and only load trusted files.

Continue with `practice.md`; use `solution.md` after attempting the exercises.